# S6E10 · Embedding MLP on original-data features

A neural network to complement gradient-boosted trees. It uses the same feature set as my tree models:

1. **Original-dataset target statistics.** The competition data is synthesised from a public CC0 survey
   dataset (129,880 rows). For every column, pair of key columns and full rating vector, I look up the
   satisfaction rate among *original* passengers with the same values. The original rows are disjoint
   from the competition data, so this cannot leak competition labels.
2. **Nearest original neighbours.** Synthetic rows are perturbed copies of original rows, so the labels of
   the closest original passengers stay informative where no exact match exists (one feature alone has AUC ≈ 0.95).
3. **Frequency encoding** of numeric values and of the full rating vector (label-free).

The MLP embeds every low-cardinality column (the 0–5 ratings and the 4 categoricals) and feeds standardised
numeric features alongside. Same 5-fold `StratifiedKFold(seed=42)` as my local models, so the OOF predictions
can be blended with them.


In [ ]:
import glob, os, time
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from pathlib import Path

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found under /kaggle/input"
    return hits[0]

COMP = Path(find("sample_submission.csv")).parent
ORIG_CSV = find("data.csv")
DATA = COMP
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(COMP, ORIG_CSV, DEVICE)


## Features (shared with my tree models)

In [ ]:
"""Shared loading, feature and cross-validation helpers for Playground S6E10 (airline satisfaction)."""



import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

TARGET = "satisfaction"
N_FOLDS = 5
SEED = 42

CATEGORICAL = ["Gender", "Customer Type", "Type of Travel", "Class"]
RATINGS = [
    "Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking", "Gate location",
    "Food and drink", "Online boarding", "Seat comfort", "Inflight entertainment", "On-board service",
    "Leg room service", "Baggage handling", "Checkin service", "Cleanliness",
]


def load() -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:
    train = pd.read_csv(DATA / "train.csv")
    test = pd.read_csv(DATA / "test.csv")
    y = train.pop(TARGET).astype(int).to_numpy()
    return train, test, y


def basic_features(df: pd.DataFrame) -> pd.DataFrame:
    """Raw columns with categoricals as pandas category dtype (GBM libraries handle them natively)."""
    out = df.drop(columns=["id"]).copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype("category")
    return out


def features(df: pd.DataFrame, mode: str) -> pd.DataFrame:
    """Feature views compared in experiments.

    basic   : raw columns, only the four text columns categorical
    cat     : also treat every 0-5 rating as categorical; synthetic generators often break the ordinal
              relationship, so per-level splits can beat threshold splits
    allcat  : additionally bin Age and Flight Distance as categories, plus summary statistics of ratings
    orig    : basic + target statistics looked up from the original dataset (see original_stats)
    """
    out = basic_features(df)
    if mode == "basic":
        return out
    if mode == "orig":
        return pd.concat([out, original_stats(df, load_original())], axis=1)
    if mode in ("orig2", "knn"):
        orig = load_original()
        parts = [out, original_stats(df, orig), pair_stats(df, orig), frequency(df)]
        if mode == "knn":
            split = "train" if df["id"].iloc[0] == 0 else "test"
            parts.append(KNN[split].set_axis(df.index))
        return pd.concat(parts, axis=1)
    ratings = df[RATINGS]
    out["rating_mean"] = ratings.mean(axis=1)
    out["rating_min"] = ratings.min(axis=1)
    out["n_zero_ratings"] = (ratings == 0).sum(axis=1)
    out["delay_total"] = df["Departure Delay in Minutes"] + df["Arrival Delay in Minutes"].fillna(0)
    out["arrival_missing"] = df["Arrival Delay in Minutes"].isna().astype(int)
    for col in RATINGS:
        out[col] = df[col].astype("category")
    if mode == "allcat":
        out["Age_cat"] = df["Age"].astype("category")
        out["Distance_cat"] = (df["Flight Distance"] // 50).astype("category")
    return out


def load_original() -> pd.DataFrame:
    """The public CC0 dataset the competition data was synthesised from (129,880 survey rows)."""
    orig = pd.read_csv(ORIG_CSV)
    orig[TARGET] = orig[TARGET].astype(int)
    return orig


RATING_KEY = RATINGS
SERVICE_KEY = ["Customer Type", "Type of Travel", "Class"] + RATINGS


def original_stats(df: pd.DataFrame, orig: pd.DataFrame, smoothing: float = 20.0) -> pd.DataFrame:
    """Target statistics from the original dataset, looked up for each competition row.

    The original rows are disjoint from the competition data, so these features cannot leak the
    competition labels and are safe to compute once, outside the CV loop.
    """
    prior = orig[TARGET].mean()
    out = pd.DataFrame(index=df.index)
    raw_cols = CATEGORICAL + RATINGS + ["Age", "Flight Distance", "Departure Delay in Minutes"]
    for col in raw_cols:
        g = orig.groupby(col)[TARGET].agg(["sum", "count"])
        te = (g["sum"] + smoothing * prior) / (g["count"] + smoothing)
        out[f"orig_te_{col}"] = df[col].map(te).astype(float).fillna(prior)
    for name, key in [("ratings", RATING_KEY), ("service", SERVICE_KEY), ("full", [c for c in orig.columns if c != TARGET])]:
        g = orig.groupby(key, dropna=False)[TARGET].agg(["mean", "count"]).reset_index()
        g = g.rename(columns={"mean": f"orig_key_{name}_mean", "count": f"orig_key_{name}_n"})
        merged = df[key].merge(g, on=key, how="left")
        out[f"orig_key_{name}_mean"] = merged[f"orig_key_{name}_mean"].to_numpy()
        out[f"orig_key_{name}_n"] = merged[f"orig_key_{name}_n"].fillna(0).to_numpy()
    return out


PAIR_COLS = [
    "Online boarding", "Inflight wifi service", "Type of Travel", "Class", "Customer Type",
    "Inflight entertainment", "Seat comfort", "Leg room service",
]


def pair_stats(df: pd.DataFrame, orig: pd.DataFrame, smoothing: float = 20.0) -> pd.DataFrame:
    """Original-dataset target mean for every pair of the most important columns."""
    prior = orig[TARGET].mean()
    out = {}
    for i, a in enumerate(PAIR_COLS):
        for b in PAIR_COLS[i + 1:]:
            g = orig.groupby([a, b])[TARGET].agg(["sum", "count"])
            te = ((g["sum"] + smoothing * prior) / (g["count"] + smoothing)).rename("te").reset_index()
            out[f"orig_pair_{a}__{b}"] = df[[a, b]].merge(te, on=[a, b], how="left")["te"].fillna(prior).to_numpy()
    return pd.DataFrame(out, index=df.index)


_FREQ_CACHE: dict[str, pd.Series] = {}


def frequency(df: pd.DataFrame) -> pd.DataFrame:
    """How often each value occurs across train + test (label-free, so safe outside the CV loop)."""
    if not _FREQ_CACHE:
        both = pd.concat([pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")])
        for col in ["Age", "Flight Distance", "Departure Delay in Minutes", "Arrival Delay in Minutes"]:
            _FREQ_CACHE[col] = both[col].value_counts()
        _FREQ_CACHE["ratings_key"] = both[RATINGS].astype(str).agg("".join, axis=1).value_counts()
    out = pd.DataFrame(index=df.index)
    for col, counts in _FREQ_CACHE.items():
        values = df[RATINGS].astype(str).agg("".join, axis=1) if col == "ratings_key" else df[col]
        out[f"freq_{col}"] = values.map(counts).fillna(0).to_numpy()
    return out


def folds(y: np.ndarray):
    return StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y)


def save(name: str, oof: np.ndarray, test_pred: np.ndarray, test_ids: pd.Series, y: np.ndarray) -> float:
    OOF.mkdir(exist_ok=True)
    SUB.mkdir(exist_ok=True)
    auc = roc_auc_score(y, oof)
    np.save(OOF / f"{name}_oof.npy", oof)
    np.save(OOF / f"{name}_test.npy", test_pred)
    pd.DataFrame({"id": test_ids, TARGET: test_pred}).to_csv(SUB / f"{name}.csv", index=False)
    print(f"{name}: OOF AUC {auc:.6f}")
    return auc

KNN = {}
KS = (5, 25, 100)


def _matrix(df: pd.DataFrame) -> np.ndarray:
    parts = [df[RATINGS].to_numpy(float)]
    for col in CATEGORICAL:
        parts.append(pd.get_dummies(df[col]).reindex(columns=sorted(df[col].unique()), fill_value=0).to_numpy(float) * 2.0)
    parts.append(df[["Age"]].to_numpy(float) / 10.0)
    parts.append(np.log1p(df[["Flight Distance"]].to_numpy(float)))
    return np.hstack(parts)


def build() -> None:
    train, test, _ = load()
    orig = load_original()
    cats = {c: sorted(orig[c].unique()) for c in CATEGORICAL}
    for df in (train, test, orig):
        for c in CATEGORICAL:
            df[c] = pd.Categorical(df[c], categories=cats[c])
    X_orig = _matrix(orig)
    y_orig = orig[TARGET].to_numpy()
    nn = NearestNeighbors(n_neighbors=max(KS), n_jobs=-1).fit(X_orig)
    for name, df in (("train", train), ("test", test)):
        dist, idx = nn.kneighbors(_matrix(df))
        out = {"knn_dist1": dist[:, 0], f"knn_dist{KS[1]}": dist[:, KS[1] - 1]}
        for k in KS:
            out[f"knn_mean{k}"] = y_orig[idx[:, :k]].mean(axis=1)
        # distance-weighted vote over the closest 25
        w = 1.0 / (dist[:, :KS[1]] + 0.5)
        out["knn_wmean25"] = (y_orig[idx[:, :KS[1]]] * w).sum(axis=1) / w.sum(axis=1)
        KNN[name] = pd.DataFrame(out)
        print(name, "done", flush=True)




t = time.time(); build(); print(f'knn {time.time()-t:.0f}s')

train, test, y = load()
X, X_test = features(train, "knn"), features(test, "knn")
print(X.shape, X_test.shape)


## Encoding for the network

In [ ]:
EMB_COLS = CATEGORICAL + RATINGS
NUM_COLS = [c for c in X.columns if c not in EMB_COLS]

def encode(X, X_test):
    codes, cards = [], []
    for c in EMB_COLS:
        cats = pd.Categorical(pd.concat([X[c].astype(str), X_test[c].astype(str)])).categories
        codes.append((pd.Categorical(X[c].astype(str), cats).codes, pd.Categorical(X_test[c].astype(str), cats).codes))
        cards.append(len(cats))
    C = np.stack([a for a, _ in codes], 1).astype(np.int64)
    CT = np.stack([b for _, b in codes], 1).astype(np.int64)
    N, NT = X[NUM_COLS].astype(float).to_numpy(), X_test[NUM_COLS].astype(float).to_numpy()
    skewed = [i for i, c in enumerate(NUM_COLS) if c in ("Age", "Flight Distance", "Departure Delay in Minutes",
              "Arrival Delay in Minutes", "delay_total") or c.startswith("freq_") or c.endswith("_n")]
    N[:, skewed], NT[:, skewed] = np.log1p(np.nan_to_num(N[:, skewed])), np.log1p(np.nan_to_num(NT[:, skewed]))
    usable = ~np.isnan(N).all(0) & (np.nanstd(N, 0) > 0)  # drop all-missing / constant columns
    N, NT = N[:, usable], NT[:, usable]
    med = np.nanmedian(N, 0)
    N, NT = np.where(np.isnan(N), med, N), np.where(np.isnan(NT), med, NT)
    mu, sd = N.mean(0), N.std(0) + 1e-6
    return C, CT, ((N - mu) / sd).astype(np.float32), ((NT - mu) / sd).astype(np.float32), cards

C, CT, N, NT, CARDS = encode(X, X_test)
print(C.shape, N.shape, CARDS)


## Model: embeddings + MLP, one-cycle schedule, best epoch by validation AUC

In [ ]:
class Net(nn.Module):
    def __init__(self, cards, n_num, emb=8, hidden=(512, 256, 128), drop=0.2):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(c + 1, emb) for c in cards])
        layers, d = [], emb * len(cards) + n_num
        for h in hidden:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(drop)]
            d = h
        self.mlp = nn.Sequential(*layers, nn.Linear(d, 1))

    def forward(self, c, x):
        e = torch.cat([m(c[:, i]) for i, m in enumerate(self.embs)], 1)
        return self.mlp(torch.cat([e, x], 1)).squeeze(1)


def predict(net, c, x, bs=16384):
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(c), bs):
            out.append(torch.sigmoid(net(c[i:i+bs], x[i:i+bs])).float().cpu().numpy())
    return np.concatenate(out)


def fit_fold(tr, va, epochs=12, bs=2048, lr=2e-3, seed=0):
    torch.manual_seed(seed)
    net = Net(CARDS, N.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-5)
    steps = epochs * int(np.ceil(len(tr) / bs))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.25)
    c_tr, x_tr = torch.tensor(C[tr], device=DEVICE), torch.tensor(N[tr], device=DEVICE)
    y_tr = torch.tensor(y[tr], dtype=torch.float32, device=DEVICE)
    c_va, x_va = torch.tensor(C[va], device=DEVICE), torch.tensor(N[va], device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss()
    best, best_state = -1, None
    for ep in range(epochs):
        net.train()
        perm = torch.randperm(len(tr), device=DEVICE)
        for i in range(0, len(tr), bs):
            b = perm[i:i+bs]
            opt.zero_grad()
            loss_fn(net(c_tr[b], x_tr[b]), y_tr[b]).backward()
            opt.step(); sched.step()
        auc = roc_auc_score(y[va], predict(net, c_va, x_va))
        if auc > best:
            best, best_state = auc, {k: v.detach().clone() for k, v in net.state_dict().items()}
    net.load_state_dict(best_state)
    return net, best


## 5-fold CV × 3 seeds

In [ ]:
SEEDS = [0, 1, 2]
oof, test_pred = np.zeros(len(y)), np.zeros(len(CT))
c_te, x_te = torch.tensor(CT, device=DEVICE), torch.tensor(NT, device=DEVICE)
folds = StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(len(y)), y)
for k, (tr, va) in enumerate(folds):
    t = time.time()
    for s in SEEDS:
        net, best = fit_fold(tr, va, seed=s)
        oof[va] += predict(net, torch.tensor(C[va], device=DEVICE), torch.tensor(N[va], device=DEVICE)) / len(SEEDS)
        test_pred += predict(net, c_te, x_te) / (5 * len(SEEDS))
    print(f"fold {k}: AUC {roc_auc_score(y[va], oof[va]):.6f}  ({time.time()-t:.0f}s)")
print(f"OOF AUC {roc_auc_score(y, oof):.6f}")


In [ ]:
np.save("mlp_knn_oof.npy", oof)
np.save("mlp_knn_test.npy", test_pred)
pd.DataFrame({"id": test["id"], "satisfaction": test_pred}).to_csv("submission.csv", index=False, float_format="%.6f")
print(os.listdir("."))
